# Exercise 4: Compressing a Pretrained ECG Model


### <p style="text-align: right;">  Student Information  
<p style="text-align: right;"><b>Student Name:</b> Janita Collin</p>
<p style="text-align: right;"><b>Student ID:</b> 2213974</p>

# Focus of the exercise: 
**Apply pruning and quantization to compress the pretrained model while maintaining predictive performance.**

# Main objective --> Try to make the model size less than 1MB while maintaining the predictive performance 

Remember to complete the final task 

# Pretrained ECG Model 

In this exercise, we use a **pretrained ECG classification model** trained on the Physionet CINC 2021 dataset. It has learned features for identifying cardiac patterns, allowing faster training and better generalization.

Key points:

- Input: Multichannel ECG signals (e.g., 12 leads) and optional demographic data (age, gender).
- Architecture: SEResNet18 adapted for time-series ECG.
- Output: Multilabel classification of cardiac conditions.

Benefits:

- Faster convergence during retraining or fine-tuning.
- Strong baseline for pruning and quantization.
- Reduced risk of overfitting on smaller datasets.


In [1]:
# imports

import torch
import sys, h5py
import os
import time
import pandas as pd
from torch import nn
from scipy.io import loadmat
import numpy as np
import pickle
import logging


from torchinfo import summary
from torch.utils.data import Dataset, DataLoader


from sklearn.metrics import roc_auc_score, roc_curve, auc, average_precision_score
import matplotlib.pyplot as plt


PROJECT_ROOT = os.path.abspath("..")  # adjust if needed
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from seresnet18 import resnet18

# pruning
import torch_pruning as tp
from fvcore.nn import FlopCountAnalysis


In [2]:
# For gpu/cpu use: 

device_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using {device_count} {'gpu(s)' if device.type=='cuda' else 'cpu'}")


# variable definitions
channels = 12  # ECG channels
seq_len = 4096 # ECG sequence length
normalizetype = '0-1' # normalization type
# Classes of different cardiac conditions 
labels = ['426783006', '426177001', '164934002', '427393009', '713426002', '427084000', '59118001', '164889003', '59931005', '47665007', '445118002', '39732003', '164890007', '164909002', '270492004', '164947007', '251146004', '284470004']
classes = len(labels)
# Pre-trained model path
model_path = 'model.pth'

# Directory for test results
output_dir = 'Results'


using 1 cpu


#### First step is to load the pre-trained model

In [3]:
# Loading the pretrained 1D resnet18 model

model = resnet18(in_channel=channels, out_channel=len(labels))
model.load_state_dict(torch.load(model_path, map_location = device))
print('Loaded the model from: {}'.format(model_path))
model = model.to(device)

Loaded the model from: model.pth


#### Create testing dataset

The test data comes from the PhysioNet CinC Challenge ECG dataset, containing real clinical ECG recordings from multiple hospitals and devices. It is fully separate from the training and validation sets and is used to evaluate model performance on unseen data.

Each test sample includes:

- ECG signal (.mat): Multi-lead recordings (usually 12 leads), preprocessed and segmented into fixed-length inputs.
- Header file (.hea): Provides sampling frequency, number of leads, lead names, and patient demographics (age, gender).

Additional info:

- Age and gender are used as extra inputs to the model.
- All recordings are indexed in test.csv, which lists the file paths, sampling frequency, diagnostic labels (multi-label), and demographic data.

In [4]:
# Helper functions for loading the test dataset: 

# Data loading from matlab or H5 files
def load_data(case):
    if case.endswith('.mat'):
        x = loadmat(case)
        return np.asarray(x['val'], dtype=np.float64)
    else:
        with h5py.File(case) as f:
            x = f['ecg'][()]
        return np.asarray(x, dtype=np.float64)

def to_cpu(inputs):
    if isinstance(inputs, (tuple, list)):
        return tuple(x.cpu() for x in inputs)
    return inputs.cpu()

In [5]:
# ECG dataset definition 

class ECGDataset(Dataset):
    def __init__(self, path, transforms, seq_len, channels):
        df = pd.read_csv(path)
        self.transforms = transforms
        self.data = df['path'].tolist()
        
        self.seq_len = seq_len
        self.n_channels = channels
        self.labels = df.iloc[:, 4:].values
        self.multi_labels = [self.labels[i, :] for i in range(self.labels.shape[0])]
        
        self.age = df['age'].tolist()
        self.gender = df['gender'].tolist()
        self.fs = df['fs'].tolist()
        self.num_samples = len(self.data)
        
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, item):
        
        file_name = self.data[item]
        fs = self.fs[item]
        ecg = load_data(file_name)
        ecg = self.transforms(ecg)
        
        label = self.multi_labels[item]
        
        age = self.age[item]
        gender = self.gender[item]

        # Encode age and gender directly
        age_gender = np.zeros(3, dtype=np.float32)

        if age >= 0:
            age_gender[0] = age / 100

        if gender in ('Female', 'female', 'F', 'f'):
            age_gender[1] = 1
        elif gender in ('Male', 'male', 'M', 'm'):
            age_gender[2] = 1

        return ecg, torch.from_numpy(age_gender).float(), torch.from_numpy(label).float()

#### Create data transformers

Before being input to the model, ECG signals are processed to ensure consistent length, scaling, and format:

- ValClip: Pads or clips signals to a fixed length (e.g., 72,000 samples) so all inputs are uniform.
- Normalize: Scales each lead to a 0–1 range, reducing the effect of amplitude differences across patients or devices.
- Retype: Converts signals to float32 for compatibility with deep learning frameworks.

In [6]:
class ECGPreprocessor:
    def __init__(self, seq_len=72000, normalize_type="0-1", p=1.0):
        self.seq_len = seq_len
        self.normalize_type = normalize_type
        self.p = p

    def __call__(self, mseq):
        # Apply preprocessing with probability p
        if np.random.rand() > self.p:
            return mseq

        # --- Clip or pad ---
        if mseq.shape[1] < self.seq_len:
            padding = np.zeros((mseq.shape[0], self.seq_len - mseq.shape[1]), dtype=np.float32)
            mseq = np.hstack((mseq, padding))

        # --- Normalize ---
        if self.normalize_type == "0-1":
            for i, row in enumerate(mseq):
                if np.sum(row) != 0:
                    mseq[i, :] = (row - np.min(row)) / (np.max(row) - np.min(row))
        else:
            raise ValueError(f"Normalization type {self.normalize_type} not supported")

        # --- Convert type ---
        return mseq.astype(np.float32)


### Load and pre-process testing data 

In [7]:
# Loading the ECG test data
test_path = 'data/test_set.csv'  

transformer = ECGPreprocessor(seq_len=seq_len, normalize_type=normalizetype, p=1.0)
testing_set = ECGDataset(test_path, transformer, seq_len, channels)

print("test subset size:", len(testing_set))
test_loader = DataLoader(testing_set, batch_size=1, shuffle=False, pin_memory=(True if device == 'cuda' else False), drop_last=True)


test subset size: 204


In [8]:
# Extract a batch from the testing set for model evaluation

try:
    # Use a batch from test data
    batch = next(iter(test_loader))
    ecg_input, ag_input, _ = batch
    example_inputs = (ecg_input[:10], ag_input[:10])

except Exception:
    # Fallback: create dummy inputs
    print("Using dummy inputs for MACs calculation")
    seq_len = 4096  # adjust according to your model
    batch_size = 1
    ag_dim = 3
    example_inputs = (torch.randn(batch_size, channels, seq_len), torch.randn(batch_size, ag_dim))

### Make predictions

In [9]:
# ===============================
# TODO (Student Task 1)
# ===============================
# Run the model in evaluation mode and compute:
# 1. Predictive performance
# 2. Inference time

#
# Hint:
# - Use average_precision_score
# - Use roc_auc_score




def predict(
    model,
    test_loader,
    class_labels,
    output_dir,
    threshold=0.5
):
    """
    Run inference on a test set and compute evaluation metrics.

    """

    model.cpu().eval()
    
    os.makedirs(output_dir, exist_ok=True)

    y_true_list, y_pre_list = [], []
    inference_time = []
    drop_empty = True

    with torch.no_grad():
        
        for i, (ecgs, ag, targets) in enumerate(test_loader):

            ecgs = ecgs.to('cpu')
            ag = ag.to('cpu')
            targets = targets.to('cpu')

            start = time.time()

            logits = model(ecgs, ag)
            probs = torch.sigmoid(logits)

            inference_time.append(time.time() - start)

            y_true_list.append(targets)
            y_pre_list.append(probs)

            if i % 50 == 0:
                print(f"{i+1:>4}/{len(test_loader)} predictions made")
     

    # Concatenate once 
    y_true = torch.cat(y_true_list, dim=0)
    y_pred = torch.cat(y_pre_list, dim=0)


    # Move tensors to CPU and convert to NumPy
    true_labels = y_true.detach().cpu().numpy().astype(np.int32)
    pred_probs = y_pred.detach().cpu().numpy().astype(np.float32)

    # Binarize predictions: threshold + ensure at least argmax is 1
    pred_binary = (pred_probs >= threshold).astype(np.int32)
    pred_binary[np.arange(len(pred_probs)), np.argmax(pred_probs, axis=1)] = 1

    
    # Remove classes with no positive samples
    if drop_empty:
        empty_classes = np.where(true_labels.sum(axis=0) == 0)[0]
        if len(empty_classes) > 0:
            true_labels = np.delete(true_labels, empty_classes, axis=1)
            pred_probs = np.delete(pred_probs, empty_classes, axis=1)
            pred_binary = np.delete(pred_binary, empty_classes, axis=1)
            used_labels = np.delete(labels, empty_classes)

    # Ensure shapes match
    assert true_labels.shape[1] == pred_probs.shape[1] == pred_binary.shape[1] == len(used_labels)


    # ---- Metrics ----

    # TODO - Calculate metrics 

    test_macro_avg_prec = average_precision_score(true_labels, pred_probs, average="macro")             # <-- fill this
    test_micro_avg_prec = average_precision_score(true_labels, pred_probs, average="micro")             # <-- fill this
    test_macro_auroc = roc_auc_score(true_labels, pred_probs, average="macro")               # <-- fill this
    test_micro_auroc = roc_auc_score(true_labels, pred_probs, average="micro")               # <-- fill this


    # ---- ROC curves ----
    fpr, tpr, roc_auc = {}, {}, {}
    dir_name="roc"

    # Per-class ROC
    for i, cls in enumerate(used_labels):
        fpr[i], tpr[i], _ = roc_curve(true_labels[:, i], pred_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Micro-average ROC
    fpr["micro"], tpr["micro"], _ = roc_curve(true_labels.ravel(), pred_probs.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    # Macro-average ROC (interpolated)
    fpr_grid = np.linspace(0, 1, 1000)
    mean_tpr = np.mean([np.interp(fpr_grid, fpr[i], tpr[i]) for i in range(len(used_labels))], axis=0)
    fpr["macro"], tpr["macro"] = fpr_grid, mean_tpr
    roc_auc["macro"] = auc(fpr_grid, mean_tpr)

    # Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle("ROC Curves")

    # Micro and Macro curves
    ax1.plot(fpr["micro"], tpr["micro"], label=f"Micro-average (AUC={roc_auc['micro']:.2f})")
    ax1.plot(fpr["macro"], tpr["macro"], label=f"Macro-average (AUC={roc_auc['macro']:.2f})")

    # Per-class curves
    for i, cls in enumerate(used_labels):
        ax2.plot(fpr[i], tpr[i], label=f"{cls} (AUC={roc_auc[i]:.2f})")

    # Formatting
    for ax in (ax1, ax2):
        ax.plot([0, 1], [0, 1], "k--")  # Diagonal reference
        ax.set(xlim=(0, 1), ylim=(0, 1.05),
               xlabel="False Positive Rate", ylabel="True Positive Rate")
        ax.legend(fontsize=7)

    fig.tight_layout()

    # Save figure
    filename = f"{dir_name}roc-test.png"
    plt.savefig(os.path.join(output_dir, filename), bbox_inches="tight")
    plt.close(fig)


    # ---- Inference time ----
    avg_inference_time = np.mean(inference_time)  # <-- fill this

    return {
        "macro_ap": test_macro_avg_prec,
        "micro_ap": test_micro_avg_prec,
        "macro_auroc": test_macro_auroc,
        "micro_auroc": test_micro_auroc,
        "avg_inference_time_ms": avg_inference_time * 1000
    }


In [10]:
# Make predictions: 

metrics_before = predict(
    model=model,
    test_loader=test_loader,
    class_labels=labels,
    output_dir=output_dir,
    threshold=0.5
)

   1/204 predictions made
  51/204 predictions made
 101/204 predictions made
 151/204 predictions made
 201/204 predictions made


In [11]:
# show the model architecture before pruning
print("Model architecture before pruning:")
example_inputs = to_cpu(example_inputs)
example_ecg, example_ag = example_inputs
summary(model, input_data=(example_ecg[:1], example_ag[:1]))

Model architecture before pruning:


Layer (type:depth-idx)                        Output Shape              Param #
ResNet                                        [1, 18]                   --
├─Conv1d: 1-1                                 [1, 64, 2048]             11,520
├─BatchNorm1d: 1-2                            [1, 64, 2048]             128
├─ReLU: 1-3                                   [1, 64, 2048]             --
├─MaxPool1d: 1-4                              [1, 64, 1024]             --
├─Sequential: 1-5                             [1, 64, 1024]             --
│    └─BasicBlock: 2-1                        [1, 64, 1024]             --
│    │    └─Conv1d: 3-1                       [1, 64, 1024]             28,672
│    │    └─BatchNorm1d: 3-2                  [1, 64, 1024]             128
│    │    └─ReLU: 3-3                         [1, 64, 1024]             --
│    │    └─Dropout: 3-4                      [1, 64, 1024]             --
│    │    └─Conv1d: 3-5                       [1, 64, 1024]             28,672
│    │

In [12]:
# Model evaluation before pruning: 

# ===============================
# TODO (Student Task 2)
# ===============================
# Run the model in evaluation mode and compute:
# 1. Number of parameters
# 2. MACs
# 3. FLOPs
#
# Hint:
# - Use tp.utils.count_params 
# - Use tp.utils.count_ops_and_params
# - Use fvcore.nn.FlopCountAnalysis
# - Use os.path.getsize

print("\033[1mModel evaluation before pruning:\033[0m")

model.cpu().eval()
example_inputs = to_cpu(example_inputs)

# TODO: Count parameters
base_nparams = tp.utils.count_params(model)  # <-- fill this

# TODO: Count MACs
base_macs,_ = tp.utils.count_ops_and_params(model,example_inputs)  # <-- fill this

# TODO: Count FLOPs
logging.getLogger("fvcore").setLevel(logging.ERROR)
with torch.no_grad():
    base_fvcore_flops = FlopCountAnalysis(model, example_inputs).total()  # <-- fill this

print(f"Params before pruning: {base_nparams / 1e6:.2f} M")
print(f"MACs before pruning:   {base_macs / 1e9:.2f} G")
print(f"FLOPs before pruning:  {base_fvcore_flops / 1e9:.2f} G")

# TODO: Count Model size on disk
print(f"Model size on disk before pruning: {os.path.getsize(model_path) / 1e6:.2f} MB") # <-- fill this



# TODO: Print predictive performance and inference time
print(
        f"macro AP: {metrics_before['macro_ap'] :.2f} | "                # <-- fill this
        f"micro AP: {metrics_before['micro_ap'] :.2f} | "                # <-- fill this
        f"macro AUROC: {metrics_before['macro_auroc'] :.2f} | "             # <-- fill this
        f"micro AUROC: {metrics_before['micro_auroc'] :.2f} | "             # <-- fill this
        f"average inference time (ms): {metrics_before['avg_inference_time_ms']:.2f}" # <-- fill this
)


Model evaluation before pruning:
Params before pruning: 8.83 M
MACs before pruning:   1.61 G
FLOPs before pruning:  1.61 G
Model size on disk before pruning: 35.42 MB
macro AP: 0.64 | micro AP: 0.85 | macro AUROC: 0.95 | micro AUROC: 0.98 | average inference time (ms): 24.59


In [13]:
def get_original_channels(model):
    return {
        name: m.out_channels
        for name, m in model.named_modules()
        if isinstance(m, nn.Conv1d)
    }
    
original_layer_channels = get_original_channels(model)


# Model Pruning

Model pruning is a neural network compression technique that removes redundant weights, neurons, or channels to reduce model size, memory usage, and computational cost. It is particularly valuable for Edge AI and embedded systems, where resources like memory, energy, and latency are limited. By pruning, models become more efficient and deployable on resource-constrained devices, while the main challenge is to retain predictive performance despite the reduction in parameters.

---
## Pruning concepts: 

### 1- Pruning Granularity

Pruning can be applied at different levels of granularity (unstructured or structured). Unstructured pruning removes individual weights, creating sparse matrices with high compression potential but limited speedup on standard hardware. Structured pruning removes entire channels, filters, neurons, or blocks, maintaining dense matrices for faster, hardware-friendly inference, though with slightly less flexibility than unstructured pruning. It can be applied at **once** using a fixed pruning ratio (**one-shot pruning**) which is fast but may harm the predictive performance or **gradually over multiple steps** which leads to better performance.  


### 2- Pruning Strategies (Which weights to Remove)

Several criteria can be used to select weights or structures to prune:

#### 🔹 Magnitude-based Pruning (in unstructured pruning)
- Prunes weights with the smallest absolute values
- Simple and widely used
- Assumes small weights contribute less to the output

#### 🔹 Norm-based Pruning (in structured pruning)
- Uses L1 or L2 norms of filters or channels
- Common for structured pruning

#### 🔹 Gradient / Taylor-based Pruning
- Uses first-order or second-order Taylor expansion
- Estimates the **impact of removing a parameter on the loss**
- More accurate but computationally expensive

---

### 3- Pruning Ratio (Sparsity Level)

The **pruning ratio** (also called sparsity) defines how much of the model is removed.

$$\text{Pruning Ratio} =  \frac{\text{Number of pruned parameters}}{\text{Total parameters}}$$


#### Examples:
- 30% pruning → 70% of parameters remain
- 70% pruning → aggressive compression, higher risk of accuracy loss

Higher pruning ratios usually require:
- Gradual pruning
- Fine-tuning or retraining
- Careful layer-wise control

---



### In this lab, you will: 

- Apply **structured pruning**
- Experiment with different **pruning ratios** and different **strategies** and in both **one-shot** and **iterative** schedules
- Observe the impact on:
  - Model size / number of parameters
  - MACs / FLOPs
  - Predictive performance
- Analyze the trade-off between **compression and performance**


## Pruning

First, let's explore the model layers (prunable and unprunable layers)

In [14]:
# Helper functions
# Explore model layers 
def get_coarse_prunable_layers(list_layers = None):
    """ 
    Identifies prunable layers of the model 
    """
        
    model.cpu().eval()
        
    # Identify layers that should not be pruned: 
    prunable_layers = []
    ignored_layers = []
    layer_channels = {}
    
    for param in model.parameters():
        param.requires_grad_(True)
            
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv1d, nn.Linear)):
            if isinstance(module, nn.Linear) and module.out_features == classes:
                print(f"{name:20s} -> {str(tuple(module.weight.shape)):20s} -> Ignored layer")
                ignored_layers.append(module)
            else:
                print(f"{name:20s} -> {str(tuple(module.weight.shape)):20s} -> Prunable layer")
                prunable_layers.append((module, name))
                if isinstance(module, nn.Conv1d):
                    layer_channels[name] = module.out_channels
                    
    return prunable_layers, ignored_layers, layer_channels

# Optional :
print("Model's layers: ")
prunable_layers, ignored_layers, layer_channels  = get_coarse_prunable_layers()

Model's layers: 
conv1                -> (64, 12, 15)         -> Prunable layer
layer1.0.conv1       -> (64, 64, 7)          -> Prunable layer
layer1.0.conv2       -> (64, 64, 7)          -> Prunable layer
layer1.0.se.fc.0     -> (4, 64)              -> Prunable layer
layer1.0.se.fc.2     -> (64, 4)              -> Prunable layer
layer1.1.conv1       -> (64, 64, 7)          -> Prunable layer
layer1.1.conv2       -> (64, 64, 7)          -> Prunable layer
layer1.1.se.fc.0     -> (4, 64)              -> Prunable layer
layer1.1.se.fc.2     -> (64, 4)              -> Prunable layer
layer2.0.conv1       -> (128, 64, 7)         -> Prunable layer
layer2.0.conv2       -> (128, 128, 7)        -> Prunable layer
layer2.0.se.fc.0     -> (8, 128)             -> Prunable layer
layer2.0.se.fc.2     -> (128, 8)             -> Prunable layer
layer2.0.downsample.0 -> (128, 64, 1)         -> Prunable layer
layer2.1.conv1       -> (128, 128, 7)        -> Prunable layer
layer2.1.conv2       -> (128, 128, 7)

In [15]:
# ===============================
# TODO : Choose pruning settings
# ===============================


# 1. Experiment with different PRUNING_RATIO and observe sparsity vs AUROC. At least use three PRUNING_RATIOs
# 2. Experiment with different PRUNING_STRATEGY. (at least use two PRUNING_STRATEGY)
# 3. Explore iterative pruning vs one_shot. Set `iterative_steps=4` and `PRUNING_SCHEDULE='iterative'` and prune gradually.


# Options: "l1", "l2", "random"
PRUNING_STRATEGY = "L1"

# Options: "global", "local"
GRANULARITY = "local"

# Pruning ratio between 0 and 1 (e.g., 0.3 = prune 30%)
PRUNING_RATIO = 0.8

# Options: "one_shot", "iterative"
PRUNING_SCHEDULE = "iterative"

if PRUNING_SCHEDULE == "iterative": 
    iterative_steps = 4 
else : iterative_steps = 1

model_savepath = f"pruned_model_{int(PRUNING_RATIO*100)}_{GRANULARITY}_{PRUNING_SCHEDULE}_{PRUNING_STRATEGY}.pth"

In [16]:
# ===============================
# TODO (Student Task 3)
# ===============================
# Create a pruner object using Torch-Pruning
#
# Implement:
# - pruning strategy (e.g., L1, random)
# - dependency graph
# - granularity (eg., global pruning)
#
# Hint:
# - Use tp.pruner.MetaPruner or equivalent
# - Use tp.DependencyGraph().build_dependency

def get_pruner(strategy, model, example_inputs, granularity, iterative_steps, pruning_rate, ignored_layers):
    
    # Select importance criterion
    if strategy == "L1":
        importance = tp.importance.MagnitudeImportance(p=1)             # <-- fill this
    elif strategy == "L2":
        importance = tp.importance.MagnitudeImportance(p=2)             # <-- fill this
    elif strategy == "Rand":
        importance = tp.importance.RandomImportance()             # <-- fill this
    else:
        raise ValueError(f"Unsupported pruner strategy: {strategy}")
    
    # Build the dependency graph
    DG = tp.DependencyGraph().build_dependency(model, example_inputs=example_inputs)                         # <-- fill this
    
    # Create pruner without passing dependency_graph
    
    
    global_pruning = (granularity == "global")
    
    pruner_obj = tp.pruner.MetaPruner(
        model=model,
        example_inputs=example_inputs,
        importance=importance,
        iterative_steps=iterative_steps,
        pruning_ratio=pruning_rate,
        global_pruning=global_pruning,
        ignored_layers=ignored_layers
    )
    
    # Attach DG to the pruner (optional, needed for step if you want)
    pruner_obj.dependency_graph = DG
    
    return pruner_obj


In [17]:
# ===============================
# TODO (Student Task 4)
# ===============================
# Apply pruning iteratively / in one shot
# - Call pruner.step() for running pruning
# - Use same methods to calculate cpmpression metrics as before pruning
# - Observe how the model changes


def coarse_grained_pruning(model, example_inputs, schedule, iterative_steps):

    model.cpu().eval() # for post-training processing use cpu 
        
    print(
            f"Pruning with Coarse grained | {GRANULARITY} Granularity | "
            f"{PRUNING_STRATEGY.upper()} Strategy | {PRUNING_SCHEDULE} Schedule | "
            f"Pruning rate: {PRUNING_RATIO} in  iterative step {iterative_steps}"
    )

    channel_groups = {}
    nparams =  macs =  flops = 0

    pruner_obj = get_pruner(PRUNING_STRATEGY, model, example_inputs, GRANULARITY, iterative_steps, PRUNING_RATIO, ignored_layers)

    #tp.utils.print_tool.before_pruning(model)

    for i in range(iterative_steps):
        print(f"[Step {i+1}/{iterative_steps}] Running pruning...\n")          
        
        pruner_obj.step()              #run one pruning step           # <-- fill this

        if (i + 1 == iterative_steps) : 
            
            print("\033[1mModel evaluation after %d/%d pruning:\033[0m\n" % (i+1, iterative_steps))

            # ---- Parameter count ----
            current_size = tp.utils.count_params(model)                              # <-- fill this
            print(f"Current params:  {current_size}")

            # ---- MACs & Params ----
            macs, nparams = tp.utils.count_ops_and_params(model,example_inputs)                             # <-- fill this
            
            print("Params after Iter %d/%d of pruning: %.2f M → %.2f M"
            % (i+1, iterative_steps, base_nparams / 1e6, nparams / 1e6)
            )
            print("MACs after Iter %d/%d of pruning:   %.2f G → %.2f G"
            % (i+1, iterative_steps, base_macs / 1e9, macs / 1e9)
            )

            # ---- FLOPs (fvcore) ----
            # Silence fvcore logger
            logging.getLogger("fvcore").setLevel(logging.ERROR)
            with torch.no_grad():
                fvcore_flops = FlopCountAnalysis(model, example_inputs).total()                             # <-- fill this
            
            print("FLOPs after Iter %d/%d of pruning:  %.2f G → %.2f G"
            % (
                i+1,
                iterative_steps,
                base_fvcore_flops / 1e9,
                fvcore_flops / 1e9,
            ))

    torch.save(model, model_savepath)
    print(f"Model size on disk after pruning: {os.path.getsize(model_savepath) / 1e6:.2f} MB")       # <-- fill this
    print(f'Pruned model saved to: {model_savepath}\n')


# Apply pruning
coarse_grained_pruning(model, example_inputs, PRUNING_SCHEDULE, 4)

Pruning with Coarse grained | local Granularity | L1 Strategy | iterative Schedule | Pruning rate: 0.8 in  iterative step 4
[Step 1/4] Running pruning...

[Step 2/4] Running pruning...

[Step 3/4] Running pruning...

[Step 4/4] Running pruning...

Model evaluation after 4/4 pruning:

Current params:  353622
Params after Iter 4/4 of pruning: 8.83 M → 0.35 M
MACs after Iter 4/4 of pruning:   1.61 G → 0.07 G
FLOPs after Iter 4/4 of pruning:  1.61 G → 0.07 G
Model size on disk after pruning: 1.50 MB
Pruned model saved to: pruned_model_80_local_iterative_L1.pth



In [18]:
pruning_output_dir = f"Results_pruning_{int(PRUNING_RATIO*100)}_{GRANULARITY}_{PRUNING_SCHEDULE}_{PRUNING_STRATEGY}"

# Make predictions with pruned model
metrics_after = predict(
    model=model,
    test_loader=test_loader,
    class_labels=labels,
    output_dir=pruning_output_dir,
    threshold=0.5
)

# TODO: Count Model size on disk
print(f"Model size on disk before pruning: {os.path.getsize(model_savepath) / 1e6:.2f} MB") # <-- fill this


# TODO: Print predictive performance and inference time after pruning
print(
        f"macro AP: {metrics_after['macro_ap'] :.2f} | "                # <-- fill this
        f"micro AP: {metrics_after['micro_ap'] :.2f} | "                # <-- fill this
        f"macro AUROC: {metrics_after['macro_auroc'] :.2f} | "             # <-- fill this
        f"micro AUROC: {metrics_after['micro_auroc'] :.2f} | "             # <-- fill this
        f"average inference time (ms): {metrics_after['avg_inference_time_ms']:.2f}" # <-- fill this
)

   1/204 predictions made
  51/204 predictions made
 101/204 predictions made
 151/204 predictions made
 201/204 predictions made
Model size on disk before pruning: 1.50 MB
macro AP: 0.34 | micro AP: 0.46 | macro AUROC: 0.81 | micro AUROC: 0.80 | average inference time (ms): 5.54


In [19]:
# show the model architecture before pruning
print("Model architecture after pruning:")
example_inputs = to_cpu(example_inputs)
example_ecg, example_ag = example_inputs
summary(model, input_data=(example_ecg[:1], example_ag[:1]))

Model architecture after pruning:


Layer (type:depth-idx)                        Output Shape              Param #
ResNet                                        [1, 18]                   --
├─Conv1d: 1-1                                 [1, 12, 2048]             2,160
├─BatchNorm1d: 1-2                            [1, 12, 2048]             24
├─ReLU: 1-3                                   [1, 12, 2048]             --
├─MaxPool1d: 1-4                              [1, 12, 1024]             --
├─Sequential: 1-5                             [1, 12, 1024]             --
│    └─BasicBlock: 2-1                        [1, 12, 1024]             --
│    │    └─Conv1d: 3-1                       [1, 12, 1024]             1,008
│    │    └─BatchNorm1d: 3-2                  [1, 12, 1024]             24
│    │    └─ReLU: 3-3                         [1, 12, 1024]             --
│    │    └─Dropout: 3-4                      [1, 12, 1024]             --
│    │    └─Conv1d: 3-5                       [1, 12, 1024]             1,008
│    │    └

In [20]:
pruned_channels = {
    name: m.out_channels
    for name, m in model.named_modules()
    if isinstance(m, nn.Conv1d)
}

In [21]:
sparsity_per_layer = {}

print("Sparsity per Conv1D layer:")
for name, original in layer_channels.items():
    remaining = pruned_channels.get(name, original)

    sparsity = 1.0- (remaining/original)                                  # <-- compute this
    sparsity_per_layer[name] = sparsity

    print(
        f"{name:35s} | "
        f"{original:4d} → {remaining:4d} | "
        f"sparsity = {sparsity:.2f}"
    )


total_original = sum(layer_channels.values())                                       # <-- compute this
total_remaining = sum(pruned_channels.get(name,ch) for name, ch in layer_channels.items())                                      # <-- compute this

model_sparsity = 1.0- (total_remaining/total_original)                                       # <-- compute this

print(f"\nOverall model sparsity: {model_sparsity:.2f}")



Sparsity per Conv1D layer:
conv1                               |   64 →   12 | sparsity = 0.81
layer1.0.conv1                      |   64 →   12 | sparsity = 0.81
layer1.0.conv2                      |   64 →   12 | sparsity = 0.81
layer1.1.conv1                      |   64 →   12 | sparsity = 0.81
layer1.1.conv2                      |   64 →   12 | sparsity = 0.81
layer2.0.conv1                      |  128 →   25 | sparsity = 0.80
layer2.0.conv2                      |  128 →   25 | sparsity = 0.80
layer2.0.downsample.0               |  128 →   25 | sparsity = 0.80
layer2.1.conv1                      |  128 →   25 | sparsity = 0.80
layer2.1.conv2                      |  128 →   25 | sparsity = 0.80
layer3.0.conv1                      |  256 →   51 | sparsity = 0.80
layer3.0.conv2                      |  256 →   51 | sparsity = 0.80
layer3.0.downsample.0               |  256 →   51 | sparsity = 0.80
layer3.1.conv1                      |  256 →   51 | sparsity = 0.80
layer3.1.conv2       

- Observe the impact on:
  - Model size / number of parameters
  - MACs / FLOPs
  - Predictive performance
- Analyze the trade-off between **compression and performance**

The higher the pruning rates the smaller the model on disk. Global pruning keeps more parameters than local that aggressively reduces parameters. Also extreme pruning 0.8 can almost remove all weights which shrinks the model size but harms the performance

Lower FLOPs means faster interference and lower energy usage. Redusing the parameters directly reduces the MACs/FLOPs. Also global pruning is slightly more conservative on MACs because local pruning aggressively cuts computation. Extreme pruning can also reduce MACs almost to half or less of the original

L2 + global pruning preserves performance the best. random pruning drops all metrics. extreme pruning (0.8) collapses AP metrics, even if AUROC remains moderate. one shot pruning at high rates is particulary harmful compared to iterative prunning.

so the best compressed model that has 1MB target size, preservinving predictive performance would have 
-global granularity
-L2 pruning
-iterative schedule 
-moderate rate 0.3


## Retraining

Usually after pruning, the model is retrained for few epochs (5-10 epochs) to recover the predictive performance lost by removing the weights. As training data is not provided, we recommend that you use (reload) the pruned model obtained with the following configuration for the rest of this exercise: 

- Granularity: Local
- Strategy: L1 
- Pruning_ratio:80
- Schedule: iterative

In [22]:
# Loading the pruned resnet18 model

pruned_model = resnet18(in_channel=channels, out_channel=len(labels))
pruned_model_path = "pruned_model_80_local_iterative_L1.pth"              # <-- fill this

pruned_model = torch.load(pruned_model_path, map_location="cpu",weights_only=False)                   # <-- fill this

print('Loaded the pruned model from: {}'.format(pruned_model_path))
pruned_model = pruned_model.to('cpu')

# show the model architecture after pruning
print("Pruned Model architecture:")
example_inputs = to_cpu(example_inputs)
example_ecg, example_ag = example_inputs
summary(pruned_model, input_data=(example_ecg[:1], example_ag[:1]))

Loaded the pruned model from: pruned_model_80_local_iterative_L1.pth
Pruned Model architecture:


Layer (type:depth-idx)                        Output Shape              Param #
ResNet                                        [1, 18]                   --
├─Conv1d: 1-1                                 [1, 12, 2048]             2,160
├─BatchNorm1d: 1-2                            [1, 12, 2048]             24
├─ReLU: 1-3                                   [1, 12, 2048]             --
├─MaxPool1d: 1-4                              [1, 12, 1024]             --
├─Sequential: 1-5                             [1, 12, 1024]             --
│    └─BasicBlock: 2-1                        [1, 12, 1024]             --
│    │    └─Conv1d: 3-1                       [1, 12, 1024]             1,008
│    │    └─BatchNorm1d: 3-2                  [1, 12, 1024]             24
│    │    └─ReLU: 3-3                         [1, 12, 1024]             --
│    │    └─Dropout: 3-4                      [1, 12, 1024]             --
│    │    └─Conv1d: 3-5                       [1, 12, 1024]             1,008
│    │    └

In [23]:
# Loading the pretrained 1D resnet18 model
original_model_path = 'model.pth'
original_model = resnet18(in_channel=channels, out_channel=len(labels))
original_model.load_state_dict(torch.load(original_model_path, map_location = device))
print('Loaded the original model from: {}'.format(original_model_path))
original_model = original_model.to(device)

Loaded the original model from: model.pth


# Model quantization

Model Quantization is the process of reducing the numerical precision of a model’s weights and/or activations. The aim is to reduce the model size which lower the memory footpring, faster inference and allows for deployment on edge devices with limited resources but with potential drop in predictive performance.

Common precisions:

- FP32 (full precision) --> models are in FP32 by default 
- FP16 (half precision)
- INT8 (integer) --> default configuration for quantization

## Types of quantization: 
- Weight-only: 	Only the model’s weights are quantized. Activations remain FP32.
- Activation-only:	Only activations are quantized.
- Full (weights + activations):	Both weights and activations are quantized.

Based on the time of application of quantization, it can be considered as Quantization Aware-Training that happens while the model being trained and Post-training Quantization which happens after the model has been trained. 


### In this lab, you will: 

- Apply **different types of quantization** for post training quantization
- Experiment with different **precisions** for weights and activations using **dynamic quantization** 
- Observe the impact on:
  - Model size 
  - Predictive performance
  - Inference time
- Analyze the trade-off between **compression and performance**

In [ ]:
import torchao.quantization.quant_api as qa

print([name for name in dir(qa) if "Config" in name])

# here try to import different configurations for quantization as shown below with Int8WeightOnlyConfig and experiment with
from torchao.quantization.quant_api import quantize_, Int8WeightOnlyConfig
from torchao.quantization.quant_api import quantize_, Int8DynamicActivationInt4WeightConfig
from torchao.quantization.quant_api import quantize_,  Int8DynamicActivationInt8WeightConfig
from torchao.quantization.quant_api import Int8DynamicActivationInt8WeightConfig

import copy

['AOBaseConfig', 'Float8DynamicActivationFloat8SemiSparseWeightConfig', 'Float8DynamicActivationFloat8WeightConfig', 'Float8DynamicActivationInt4WeightConfig', 'Float8MMConfig', 'Float8StaticActivationFloat8WeightConfig', 'Float8WeightOnlyConfig', 'FqnToConfig', 'GemliteUIntXWeightOnlyConfig', 'Int4WeightOnlyConfig', 'Int8DynamicActivationInt4WeightConfig', 'Int8DynamicActivationInt8WeightConfig', 'Int8DynamicActivationIntxWeightConfig', 'Int8StaticActivationInt8WeightConfig', 'Int8WeightOnlyConfig', 'IntxWeightOnlyConfig', 'ModuleFqnToConfig', 'UIntXWeightOnlyConfig']


In [31]:
# Don't forget to copy the model to use different configurations for quantization and compare the results

quantized_model = copy.deepcopy(original_model)            # <-- fill this
quantized_pruned_model = copy.deepcopy(pruned_model)           # <-- fill this

In [44]:
# Model quantization: (First with original model)

# ===============================
# TODO (Student Task 5)
# ===============================
# Choose the quantization configuration 
# Apply quantization using the selected configuration on the original model 
# Run the model in evaluation mode and save it 
# Repeat same thing using the selected configuration on the pruned model 

#
# Hint:
# - Use quantize_ 


# Select configuration
config = Int8DynamicActivationInt8WeightConfig()   # <-- fill this with different configurations

# Quantization of original model
# Apply quantization with the selected quantization config on the original model
quantized_model.eval()
quantize_(quantized_model, config)  # <-- fill this

# Save the quantized model
quantized_model_path= "quantized_original_model_int8.pth"  # <-- fill this
torch.save(quantized_model.state_dict(),quantized_model_path)

In [45]:

# ===============================
# TODO (Student Task 6)
# ===============================
# Evaluate the quantized models
# 1. Predictive performance
# 2. Inference time
# 3. Model size
#

# Original Model evaluation after quantization: 
print("\033[1mOriginal Model evaluation after quantization:\033[0m")

quantization_output_dir = f"Results_quantization{config.__class__.__name__}"  # <---- change this to keep all the results from different experiments

# Make predictions with quantized model

metrics_after_quantization = predict(
    model=quantized_model,
    test_loader=test_loader,
    class_labels=labels,
    output_dir=quantization_output_dir,
    threshold=0.5
)


# TODO: Print predictive performance and inference time
print(
        f"macro AP: {metrics_after['macro_ap'] :.2f} | "                # <-- fill this
        f"micro AP: {metrics_after['micro_ap'] :.2f} | "                # <-- fill this
        f"macro AUROC: {metrics_after['macro_auroc'] :.2f} | "             # <-- fill this
        f"micro AUROC: {metrics_after['micro_auroc'] :.2f} | "             # <-- fill this
        f"average inference time (ms): {metrics_after['avg_inference_time_ms']:.2f}" # <-- fill this
)


Original Model evaluation after quantization:
   1/204 predictions made
  51/204 predictions made
 101/204 predictions made
 151/204 predictions made
 201/204 predictions made
macro AP: 0.34 | micro AP: 0.46 | macro AUROC: 0.81 | micro AUROC: 0.80 | average inference time (ms): 5.54


In [46]:
# ===============================
# TODO (Same as Student Task 7)
# ===============================
# Quantization of a pruned model
# Repeat same thing on the pruned model 

# Apply quantization with the selected quantization config on the pruned model
config = Int8DynamicActivationInt8WeightConfig()
quantized_pruned_model.eval()
quantize_(quantized_pruned_model,config)  # <-- fill this

# Save the quantized and pruned model
quantized_pruned_model_path="quantized_pruned_model_int8.pth"
torch.save(quantized_pruned_model.state_dict(),quantized_pruned_model_path)  # <-- fill this

In [47]:
# Pruned Model evaluation after quantization: 
print("\033[1mPruned Model evaluation after quantization:\033[0m")

quantization_pruning_output_dir = f"Results_quantization_pruning{config.__class__.__name__}" # <---- change this to keep all the results from different experiments

# Make predictions with quantized pruned model

metrics_after_QP = predict(
    model=quantized_pruned_model,
    test_loader=test_loader,
    class_labels=labels,
    output_dir=quantization_output_dir,
    threshold=0.5
)


# TODO: Print predictive performance and inference time
print(
        f"macro AP: {metrics_after_QP['macro_ap'] :.2f} | "                # <-- fill this
        f"micro AP: {metrics_after_QP['micro_ap'] :.2f} | "                # <-- fill this
        f"macro AUROC: {metrics_after_QP['macro_auroc'] :.2f} | "             # <-- fill this
        f"micro AUROC: {metrics_after_QP['micro_auroc'] :.2f} | "             # <-- fill this
        f"average inference time (ms): {metrics_after_QP['avg_inference_time_ms']:.2f}" # <-- fill this
)


Pruned Model evaluation after quantization:
   1/204 predictions made
  51/204 predictions made
 101/204 predictions made
 151/204 predictions made
 201/204 predictions made
macro AP: 0.34 | micro AP: 0.46 | macro AUROC: 0.81 | micro AUROC: 0.80 | average inference time (ms): 14.05


In [48]:
# TODO: compare between pruned model, original model and quantized models

print(f"Model size on disk before pruning (baseline): {os.path.getsize(original_model_path) / 1e6:.2f} MB") # <-- fill this

print(f"Model size on disk after pruning (pruned model): {os.path.getsize(pruned_model_path) / 1e6:.2f} MB") # <-- fill this

# TODO: Count Model size on disk
print(f"Model size on disk after quantization: {os.path.getsize(quantized_model_path) / 1e6:.2f} MB") # <-- fill this

# TODO: Count Model size on disk
print(f"Model size on disk after pruning and quantization: {os.path.getsize(quantized_pruned_model_path) / 1e6:.2f} MB") # <-- fill this

Model size on disk before pruning (baseline): 35.42 MB
Model size on disk after pruning (pruned model): 1.50 MB
Model size on disk after quantization: 35.17 MB
Model size on disk after pruning and quantization: 1.47 MB


Experiments with different quantization configurations and interpret the results
    INT8 weight only quantization didn't reduce the model size much (35.42MB -> 35.17MB) and it led to a large drop in predictive performance for the original model (macro AP: 0.11, micro AP: 0.11). Dynamic quantization with INT8 activations and INT4 weights preserved predictive performance (macro AP:0.34, micro AP: 0.34, macro AUROC:0.81, micro AUROC:0.80) with similar interference time about 5.5 ms for the original model.The Full INT8 quantization (INT8 weights and INT8 activations) also preserved predictive performance with the same metrics, but showed a litle higher inference time when applied to the pruned model

Experiments with different combinations of quantization and pruning configurations and interpret the results(what does this mean? that I shoulhave tested quantization with other pruned models than what was said in the assignment(the 80%, L1, local,iterative) This question is difficult to understand what should I be intreperenting this is probably because I have dyslexia but I will intrepret here the configuration about quantization tu the 80%, L1, local,iterative pruned model)
    The structured iterative prunning with an 80% pruning ratio reduced the model size from 35.42MB to approximately 1.5 MB without retraining, but still preserving the predictive performance (macro AP:0.34, micro AP: 0.46). Combining pruning with quantization further reduced the size of the model to 1.47MB while maintaining the predictive performance which shows that pruning is the dominatt factor for model compression while quantization helps to maintain performance and improves deployment effiency.

# Final task (compressed model)
In this final task, we expect the student to provide a final compressed model for which the model size is around 1MB and the predictive performance is preserved. Write down the configurations used for pruning and quantization in a way to allow reproducing the results and remember to attach the resulting model with your submission. 

In [57]:
# ===============================
# TODO (FINAL Student Task 8)
# ===============================
# Provide the final configuration for pruning 
# Provide the final configuration for quantization 
# Save the final model (with 1MB as model size and preserved predictive performance) and return it 


# Example: 

# Pruning configuration 
PRUNING_STRATEGY = "L1"
GRANULARITY = "local"
PRUNING_RATIO = 0.8
PRUNING_SCHEDULE = "iterative"
if PRUNING_SCHEDULE == "iterative": 
    iterative_steps = 4 
else : iterative_steps = 0
ignored_layers = []  # layers to ignore during pruning
#didn't the assignment say to use the L1,80,local,iterative for the rest of the exercise?
# Quantization configuration 
confi = Int8DynamicActivationInt8WeightConfig()

# Apply pruning 
pruner_objc = get_pruner(
    strategy=PRUNING_STRATEGY,
    model=model,
    example_inputs=example_inputs,
    granularity=GRANULARITY,
    iterative_steps=iterative_steps,
    pruning_rate=PRUNING_RATIO,
    ignored_layers=ignored_layers
)
for step in range(iterative_steps):
    pruner_objc.step()
prun_model= model
# Apply quantization 
combined_model= prun_model
quantize_(combined_model, confi)


# Save the quantized and pruned model
final_model_path="compressed_model_final.pth"
torch.save(combined_model.state_dict(),final_model_path) 

combined_model

ResNet(
  (conv1): Conv1d(12, 1, kernel_size=(15,), stride=(2,), padding=(7,), bias=False)
  (bn1): BatchNorm1d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool1d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv1d(1, 1, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
      (bn1): BatchNorm1d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv1d(1, 1, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
      (bn2): BatchNorm1d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (se): SELayer(
        (avg_pool): AdaptiveAvgPool1d(output_size=1)
        (fc): Sequential(
          (0): Linear(in_features=1, out_features=1, bias=False, weight=Tensor: <class 'torch.nn.parameter.Parameter'>)
          (1): ReLU(inplace=True)
          (2): Lin